# Telco Customer Churn Analysis

A single, end-to-end notebook for exploring the prepared customer data, creating EDA charts, and comparing churn models.

**Input:** `cleaned_telco.csv`  
**Outputs:** charts in `eda_visualizations/` and model scores in `model_comparison.csv`

Install project dependencies with `python -m pip install -r requirements.txt`, then run the notebook from the project folder.


## 1. Setup and load data

Load the prepared dataset and check that the columns needed for analysis are present.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "cleaned_telco.csv"
CHARTS_DIR = PROJECT_DIR / "eda_visualizations"
RESULTS_PATH = PROJECT_DIR / "model_comparison.csv"

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Place cleaned_telco.csv in the project folder: {PROJECT_DIR}")

CHARTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", palette="Set2")
data = pd.read_csv(DATA_PATH)

required_columns = {
    "Churn", "Gender", "Contract", "Internet Service",
    "Tenure in Months", "Monthly Charge",
}
missing_columns = required_columns - set(data.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

if not data["Churn"].isin([0, 1]).all():
    raise ValueError("Churn must contain only 0 (stayed) and 1 (churned).")

print(f"Loaded {len(data):,} customers and {len(data.columns)} columns.")
display(data.head())


## 2. Understand the data

Review the dataset size, data types, missing values, duplicates, and churn balance.

In [ ]:
print("Dataset shape:", data.shape)
print("Duplicate rows:", int(data.duplicated().sum()))

profile = pd.DataFrame({
    "data_type": data.dtypes.astype(str),
    "missing_values": data.isna().sum(),
    "unique_values": data.nunique(dropna=False),
})
display(profile)

churn_counts = data["Churn"].value_counts().rename(index={0: "Stayed", 1: "Churned"})
churn_summary = churn_counts.to_frame("customers")
churn_summary["share (%)"] = churn_summary["customers"].div(len(data)).mul(100)
display(churn_summary)


## 3. Exploratory data analysis

Create and save the main churn visualizations in `eda_visualizations/`.

In [ ]:
plot_data = data.copy()
plot_data["Churn status"] = plot_data["Churn"].map({0: "Stayed", 1: "Churned"})


def save_plot(filename):
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=plot_data, x="Churn status", order=["Stayed", "Churned"])
plt.title("Customer Churn")
plt.xlabel("")
plt.ylabel("Customers")
save_plot("01_churn_count.png")


In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=plot_data, x="Gender", hue="Churn status", hue_order=["Stayed", "Churned"])
plt.title("Churn by Gender")
plt.xlabel("Gender")
plt.ylabel("Customers")
save_plot("02_gender_vs_churn.png")

plt.figure(figsize=(8, 4))
sns.countplot(data=plot_data, x="Contract", hue="Churn status", hue_order=["Stayed", "Churned"])
plt.title("Churn by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Customers")
plt.xticks(rotation=15)
save_plot("03_contract_vs_churn.png")


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(
    data=plot_data,
    x="Tenure in Months",
    hue="Churn status",
    bins=30,
    element="step",
    stat="density",
    common_norm=False,
)
plt.title("Tenure Distribution by Churn")
plt.xlabel("Tenure (months)")
plt.ylabel("Density")
save_plot("04_tenure_distribution.png")

plt.figure(figsize=(7, 4))
sns.boxplot(data=plot_data, x="Churn status", y="Monthly Charge", order=["Stayed", "Churned"])
plt.title("Monthly Charges by Churn")
plt.xlabel("")
plt.ylabel("Monthly charge")
save_plot("05_monthly_charges_vs_churn.png")


In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=plot_data, x="Internet Service", hue="Churn status", hue_order=["Stayed", "Churned"])
plt.title("Churn by Internet Service")
plt.xlabel("Internet service")
plt.ylabel("Customers")
save_plot("06_internet_service_vs_churn.png")

numeric_data = data.select_dtypes(include="number")
correlation = numeric_data.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(correlation, cmap="coolwarm", center=0, linewidths=0.25)
plt.title("Correlation of Numeric Features")
save_plot("07_correlation_heatmap.png")


## 4. Churn patterns

Calculate churn rates from the dataset rather than relying on fixed observations.

In [ ]:
churn_rate = data["Churn"].mean()
print(f"Overall churn rate: {churn_rate:.1%}")

for column in ["Gender", "Contract", "Internet Service"]:
    rates = data.groupby(column, observed=True)["Churn"].mean().sort_values(ascending=False)
    highest_group = rates.index[0]
    print(f"Highest churn by {column}: {highest_group} ({rates.iloc[0]:.1%})")

median_tenure = plot_data.groupby("Churn status", observed=True)["Tenure in Months"].median()
median_charges = plot_data.groupby("Churn status", observed=True)["Monthly Charge"].median()
print("Median tenure by churn status (months):")
display(median_tenure.to_frame("months"))
print("Median monthly charge by churn status:")
display(median_charges.to_frame("monthly charge"))


## 5. Prepare and compare models

Split the data into stratified train/test sets. Fit imputation-free encoders and scaling inside each pipeline to avoid test-set leakage.

In [ ]:
X = data.drop(columns="Churn")
y = data["Churn"]

categorical_columns = X.select_dtypes(exclude="number").columns.tolist()
numerical_columns = X.select_dtypes(include="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("category", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ("number", StandardScaler(), numerical_columns),
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}


In [ ]:
results = []

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
    })

comparison = pd.DataFrame(results)
comparison.to_csv(RESULTS_PATH, index=False)
print(f"Saved model comparison to {RESULTS_PATH}")
display(comparison.style.format({
    "Accuracy": "{:.1%}",
    "Precision": "{:.1%}",
    "Recall": "{:.1%}",
    "F1-score": "{:.1%}",
}))


## 6. Wrap-up

The notebook has produced refreshed EDA charts and model evaluation scores. Use `model_comparison.csv` to view or share the model metrics outside the notebook.